#Notebook 06 - Catálogo

##1. Contexto e comentários de catálogo e schemas



In [0]:
%sql
USE CATALOG mvp_pipeline;

COMMENT ON CATALOG mvp_pipeline IS
'MVP de pipeline de dados na nuvem: incentivos fiscais de ICMS, dinâmica de pequenos negócios e crescimento econômico setorial no Rio Grande do Sul. Fontes abertas da SEFAZ-RS (Receita Dados), do IBGE e do Sebrae RS. Arquitetura medalhão com três camadas.';

COMMENT ON SCHEMA bronze IS
'Dado como veio da fonte, sem alteração de conteúdo, tudo em texto. Carrega metadados de linhagem: _ingestao_ts, _arquivo_origem, _url_origem e _fonte.';

COMMENT ON SCHEMA silver IS
'Dado limpo e padronizado: tipagem, conversão de decimal com vírgula, normalização de chaves e marcação de características da fonte. Nenhuma linha é descartada.';

COMMENT ON SCHEMA gold IS
'Modelo dimensional em constelação: seis dimensões, uma ponte N:N e sete fatos de granularidades diferentes. Regras de negócio aplicadas como colunas marcadoras, sem exclusão de dado.';

##2. Dimensão de tempo


In [0]:
%sql

COMMENT ON TABLE gold.dim_tempo IS
'Dimensão de tempo. Grão: mês. Período: 2015-01 a 2026-12. Chave: sk_tempo. Origem: gerada.';

ALTER TABLE gold.dim_tempo ALTER COLUMN sk_tempo COMMENT 'Chave da dimensão, formato AAAAMM.';
ALTER TABLE gold.dim_tempo ALTER COLUMN data_ref COMMENT 'Primeiro dia do mês.';
ALTER TABLE gold.dim_tempo ALTER COLUMN ano COMMENT 'Ano civil.';
ALTER TABLE gold.dim_tempo ALTER COLUMN trimestre COMMENT 'Trimestre civil, 1 a 4.';
ALTER TABLE gold.dim_tempo ALTER COLUMN mes COMMENT 'Mês civil, 1 a 12.';
ALTER TABLE gold.dim_tempo ALTER COLUMN nome_mes COMMENT 'Nome do mês por extenso.';
ALTER TABLE gold.dim_tempo ALTER COLUMN ano_completo COMMENT
'Ano com doze meses na série. Falso só em 2026. Filtro padrão de comparação anual.';
ALTER TABLE gold.dim_tempo ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##3. Dimensão de atividade econômica



In [0]:
%sql

COMMENT ON TABLE gold.dim_cnae IS
'Dimensão de atividade econômica. Grão: subclasse CNAE de sete dígitos. Chave: cnae_subclasse. Liga a ponte_cnae_cadeia e aos fatos de ICMS, desoneração e cadastro. Origem: códigos presentes em quatro fontes.';

ALTER TABLE gold.dim_cnae ALTER COLUMN cnae_subclasse COMMENT 'Código CNAE de subclasse, sete dígitos.';
ALTER TABLE gold.dim_cnae ALTER COLUMN nome_cnae_subclasse COMMENT 'Denominação da subclasse.';
ALTER TABLE gold.dim_cnae ALTER COLUMN versao_cnae COMMENT 'Versão da classificação: 1.1 ou 2.0.';
ALTER TABLE gold.dim_cnae ALTER COLUMN cnae_secao COMMENT 'Seção da CNAE. Nula fora das desonerações.';
ALTER TABLE gold.dim_cnae ALTER COLUMN cnae_divisao COMMENT 'Divisão da CNAE, dois dígitos.';
ALTER TABLE gold.dim_cnae ALTER COLUMN cnae_grupo COMMENT 'Grupo da CNAE, três dígitos.';
ALTER TABLE gold.dim_cnae ALTER COLUMN cnae_classe COMMENT 'Classe da CNAE, cinco dígitos.';
ALTER TABLE gold.dim_cnae ALTER COLUMN sem_cnae COMMENT
'Código residual 0000000, sem atividade atribuída. Responde por 4,16% do ICMS.';
ALTER TABLE gold.dim_cnae ALTER COLUMN tem_cadeia COMMENT
'Código com correspondência no de-para do Sebrae RS. Falso em 90 subclasses.';
ALTER TABLE gold.dim_cnae ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

In [0]:
%sql

-- Verificação até aqui

SELECT table_name, column_name, comment
FROM mvp_pipeline.information_schema.columns
WHERE table_schema = 'gold' AND table_name IN ('dim_tempo', 'dim_cnae')
ORDER BY table_name, ordinal_position;

##4. Dimensão de cadeia produtiva


In [0]:
%sql

COMMENT ON TABLE gold.dim_cadeia_produtiva IS
'Dimensão de cadeia produtiva. Grão: cadeia. Chave: cadeia. Liga a dim_cnae por ponte_cnae_cadeia, em relação N:N. Origem: classificação do Sebrae RS, 2026.';

ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN cadeia COMMENT
'Nome da cadeia. Catorze prioritárias mais "Outros (não priorizado)".';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN cadeia_prioritaria COMMENT 'Cadeia priorizada pelo Sebrae RS.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN qtd_cnaes COMMENT 'Subclasses que compõem a cadeia, de 1 a 149.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN cobertura_icms_perc COMMENT
'Percentual das subclasses da cadeia com ICMS em 2024.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN ressalva_volatilidade COMMENT 'Cadeia com menos de cinco subclasses.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN ressalva_cobertura COMMENT 'Cobertura de ICMS abaixo de 70%.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN ressalva COMMENT
'Texto da ressalva, para exibir junto do resultado. Nulo quando não há.';
ALTER TABLE gold.dim_cadeia_produtiva ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##5. Ponte entre CNAE e cadeia produtiva

In [0]:
%sql

COMMENT ON TABLE gold.ponte_cnae_cadeia IS
'Ponte N:N entre atividade econômica e cadeia produtiva. Grão: par CNAE-cadeia. 1.394 pares para 1.358 subclasses. Valores por cadeia não somam ao total: somar todas superestima em 3,02%.';

ALTER TABLE gold.ponte_cnae_cadeia ALTER COLUMN cnae_subclasse COMMENT 'Liga a dim_cnae e aos fatos.';
ALTER TABLE gold.ponte_cnae_cadeia ALTER COLUMN cadeia COMMENT 'Liga a dim_cadeia_produtiva.';
ALTER TABLE gold.ponte_cnae_cadeia ALTER COLUMN cadeia_prioritaria COMMENT 'Repetido da dimensão, para filtro sem junção.';
ALTER TABLE gold.ponte_cnae_cadeia ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##6. Dimensão de município

In [0]:
%sql

COMMENT ON TABLE gold.dim_municipio IS
'Dimensão de território. Grão: município do RS, mais dois códigos residuais da SEFAZ. Chave: cod_municipio_ibge. Chave alternativa: cod_munic_sefaz. Origem: IBGE, com COREDE do cadastro da SEFAZ-RS. O COREDE existe só aqui, em nenhum fato.';

ALTER TABLE gold.dim_municipio ALTER COLUMN cod_municipio_ibge COMMENT
'Código IBGE, sete dígitos. Nulo nos pseudomunicípios.';
ALTER TABLE gold.dim_municipio ALTER COLUMN nome_municipio COMMENT 'Nome do município, sem sufixo de UF.';
ALTER TABLE gold.dim_municipio ALTER COLUMN nome_corede COMMENT
'Conselho Regional de Desenvolvimento, recorte territorial oficial do RS.';
ALTER TABLE gold.dim_municipio ALTER COLUMN microrregiao COMMENT 'Microrregião do IBGE.';
ALTER TABLE gold.dim_municipio ALTER COLUMN mesorregiao COMMENT 'Mesorregião do IBGE.';
ALTER TABLE gold.dim_municipio ALTER COLUMN cod_munic_sefaz COMMENT
'Código próprio da SEFAZ-RS, distinto do IBGE. Resolvido por de-para na Silver.';
ALTER TABLE gold.dim_municipio ALTER COLUMN pseudo_municipio COMMENT
'Códigos residuais 0 (Sem Município) e 900 (Outras UF). Nunca entram em análise territorial.';
ALTER TABLE gold.dim_municipio ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##7. Dimensão de benefício fiscal

In [0]:
%sql

COMMENT ON TABLE gold.dim_beneficio IS
'Dimensão de benefício fiscal. Grão: dispositivo legal. Chave composta: imposto + tipo_beneficio + cod_beneficio; o código não é único isoladamente. Origem: demonstrativo de desonerações da SEFAZ-RS.';

ALTER TABLE gold.dim_beneficio ALTER COLUMN imposto COMMENT 'ICMS, IPVA ou ITCD.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN tipo_beneficio COMMENT
'Natureza jurídica do benefício. Dez valores.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN cod_beneficio COMMENT
'Código do dispositivo, sequencial dentro de imposto e tipo.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN descr_beneficio COMMENT 'Descrição do dispositivo.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN legislacao COMMENT 'Artigo e inciso do regulamento do imposto.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN finalidade COMMENT
'Objetivo da concessão, classificado pela fonte. Catorze valores.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN justificativa COMMENT 'Justificativa da concessão, divulgada desde 2024.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN integra_total_estadual COMMENT
'Benefício que entra no total do Estado. Falso para os heterônomos, publicados à parte pela fonte.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN finalidade_economica COMMENT
'Benefício de finalidade econômica. Filtro obrigatório do recorte setorial.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN evento_extraordinario COMMENT
'Dispositivo criado após a calamidade de 2024. Cinco casos, R$ 94,2 milhões, só naquele ano.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN origem_valor COMMENT
'DIRETO quando declarado pelo contribuinte, caso do crédito presumido; ESTIMATIVA nos demais.';
ALTER TABLE gold.dim_beneficio ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##8. Dimensão de categoria de contribuinte

In [0]:
%sql

COMMENT ON TABLE gold.dim_categoria_contribuinte IS
'Dimensão de porte e natureza do contribuinte. Grão: categoria. Chave: categoria. Origem: cadastro de contribuintes da SEFAZ-RS.';

ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN categoria COMMENT
'MEI, SIMPLES NACIONAL, GERAL, MICROPRODUTOR ou PRODUTOR.';
ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN e_mpe COMMENT
'Micro ou pequena empresa: MEI e Simples Nacional.';
ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN e_produtor_rural COMMENT
'Produtor rural, nunca somado às empresas.';
ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN entra_analise COMMENT
'Falso para MEI: série publicada só desde set/2024 e com baixas acumuladas.';
ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN ressalva COMMENT
'Limitação da categoria. Nulo quando não há.';
ALTER TABLE gold.dim_categoria_contribuinte ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##9. Fato de ICMS por atividade econômica

In [0]:
%sql

COMMENT ON TABLE gold.fato_icms_cnae IS
'Arrecadação de ICMS por atividade econômica. Grão: mês + versão da CNAE + subclasse + nome. Período: 2015 a 2026, com 2026 incompleto. Liga a dim_tempo e dim_cnae. Medida aditiva: valor_icms. Origem: SEFAZ-RS.';

ALTER TABLE gold.fato_icms_cnae ALTER COLUMN sk_tempo COMMENT 'Liga a dim_tempo.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN cnae_subclasse COMMENT 'Liga a dim_cnae.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN versao_cnae COMMENT
'Compõe o grão: a fonte publica CNAE 1.1 e 2.0 lado a lado.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN nome_cnae_subclasse COMMENT
'Compõe o grão: desde nov/2024 há duas categorias residuais sob o código 0000000.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN sem_cnae COMMENT 'Linha sem atividade econômica atribuída.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN valor_icms COMMENT
'Arrecadação líquida no mês, em reais correntes. Negativa em 16 linhas, por restituição e compensação.';
ALTER TABLE gold.fato_icms_cnae ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##10. Fato de arrecadação por território

In [0]:
%sql

COMMENT ON TABLE gold.fato_arrecadacao_municipio IS
'Arrecadação por território e tributo. Grão: mês + município + tributo. Período: 2016 a 2026, com 2026 incompleto. Liga a dim_tempo e dim_municipio. Medida aditiva: valor_arrecadado. Origem: SEFAZ-RS. A série de ICMS coincide com fato_icms_cnae: são cortes do mesmo agregado.';

ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN sk_tempo COMMENT 'Liga a dim_tempo.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN cod_municipio_ibge COMMENT
'Liga a dim_municipio. Nulo nos pseudomunicípios.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN cod_munic_sefaz COMMENT
'Código original da SEFAZ-RS, mantido para rastreabilidade.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN pseudo_municipio COMMENT 'Códigos residuais 0 e 900.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN tributo COMMENT 'ICMS, IPVA, ITCD ou taxas.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN valor_arrecadado COMMENT
'Arrecadação líquida no mês, em reais correntes.';
ALTER TABLE gold.fato_arrecadacao_municipio ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##11. Fatos de desoneração

In [0]:
%sql

COMMENT ON TABLE gold.fato_desoneracao_detalhada IS
'Desoneração fiscal com chave setorial e territorial. Grão: ano + dispositivo + COREDE + subclasse. Período: 2021 a 2024. Liga a dim_beneficio e dim_cnae. Medidas: valor_desonerado e qtd_empresas. Origem: SEFAZ-RS. Não somar com fato_desoneracao_agregada. A fonte adverte que valores por CNAE e COREDE indicam tendência, não grandeza.';

COMMENT ON TABLE gold.fato_desoneracao_agregada IS
'Desoneração fiscal sem chave setorial nem territorial. Grão: ano + dispositivo. Período: 2021 a 2024. Reúne o que a fonte estima globalmente (não estorno e Simples Nacional) e os benefícios de IPVA e ITCD. Não somar com fato_desoneracao_detalhada.';

ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN ano COMMENT
'Ano de apropriação do benefício, não necessariamente o da concessão.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN ano_completo COMMENT 'Ano completo na série.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN imposto COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN tipo_beneficio COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN cod_beneficio COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN nome_corede COMMENT
'COREDE, atributo degenerado: a fonte publica por COREDE, não por município.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN cnae_subclasse COMMENT 'Liga a dim_cnae.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN valor_desonerado COMMENT
'Valor bruto da desoneração no ano, em reais correntes, sem efeito líquido.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN qtd_empresas COMMENT
'Contribuintes que usufruíram do benefício na célula.';
ALTER TABLE gold.fato_desoneracao_detalhada ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN ano COMMENT 'Ano de apropriação do benefício.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN ano_completo COMMENT 'Ano completo na série.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN imposto COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN tipo_beneficio COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN cod_beneficio COMMENT 'Parte da chave para dim_beneficio.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN valor_desonerado COMMENT
'Valor bruto estimado no ano, em reais correntes.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN qtd_empresas COMMENT 'Contribuintes, quando informado.';
ALTER TABLE gold.fato_desoneracao_agregada ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##12. Fatos de cadastro de contribuintes

In [0]:
%sql

COMMENT ON TABLE gold.fato_cadastro_setor IS
'Cadastro de contribuintes por atividade econômica. Grão: mês + categoria + subclasse. Período: 2020 a 2026, com 2026 incompleto. Liga a dim_tempo, dim_categoria_contribuinte e dim_cnae. qtd_ativos é estoque; qtd_novos e qtd_baixados são fluxo. Origem: SEFAZ-RS.';

COMMENT ON TABLE gold.fato_cadastro_municipio IS
'Cadastro de contribuintes por território. Grão: mês + categoria + município. Período: 2020 a 2026, com 2026 incompleto. Mesmas medidas e ressalvas do fato por atividade econômica.';

ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN sk_tempo COMMENT 'Liga a dim_tempo.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN categoria COMMENT 'Liga a dim_categoria_contribuinte.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN cnae_subclasse COMMENT 'Liga a dim_cnae.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN setor COMMENT 'Setor econômico, classificação da SEFAZ-RS.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN area COMMENT 'Área de atividade, classificação da SEFAZ-RS.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN atividade COMMENT 'Atividade, classificação da SEFAZ-RS.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN qtd_ativos COMMENT
'Estoque de estabelecimentos ativos. Não aditivo no tempo.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN qtd_novos COMMENT
'Fluxo de aberturas no mês. Inclui migração entre categorias e setores.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN qtd_baixados COMMENT
'Fluxo de baixas no mês. No MEI vem acumulado, contrariando o dicionário da fonte.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN snapshot_ano COMMENT
'Último mês publicado do ano. Marca a fotografia anual do estoque.';
ALTER TABLE gold.fato_cadastro_setor ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN sk_tempo COMMENT 'Liga a dim_tempo.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN categoria COMMENT 'Liga a dim_categoria_contribuinte.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN cod_municipio_ibge COMMENT 'Liga a dim_municipio.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN qtd_ativos COMMENT 'Estoque de estabelecimentos ativos.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN qtd_novos COMMENT 'Fluxo de aberturas no mês.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN qtd_baixados COMMENT 'Fluxo de baixas no mês.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN snapshot_ano COMMENT 'Último mês publicado do ano.';
ALTER TABLE gold.fato_cadastro_municipio ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##13. Fato de PIB municipal

In [0]:
%sql

COMMENT ON TABLE gold.fato_pib_municipal IS
'PIB municipal. Grão: ano + município. Período: 2018 a 2023, limite de publicação do IBGE. Liga a dim_municipio. Origem: IBGE, API de Agregados v3, tabela 5938. Valores a preços correntes: não medem crescimento real.';

ALTER TABLE gold.fato_pib_municipal ALTER COLUMN ano COMMENT 'Ano de referência.';
ALTER TABLE gold.fato_pib_municipal ALTER COLUMN cod_municipio_ibge COMMENT 'Liga a dim_municipio.';
ALTER TABLE gold.fato_pib_municipal ALTER COLUMN pib_reais COMMENT 'PIB a preços correntes, em reais.';
ALTER TABLE gold.fato_pib_municipal ALTER COLUMN pib_mil_reais COMMENT
'PIB na unidade original do IBGE, mil reais, para conferência com a fonte.';
ALTER TABLE gold.fato_pib_municipal ALTER COLUMN _processamento_ts COMMENT 'Momento da última carga.';

##14. Relatório de Invariantes

In [0]:
%sql

-- Catálogo da tabela de invariantes

COMMENT ON TABLE gold.relatorio_invariantes IS
'Resultado das invariantes de negócio da camada Gold. Grão: uma verificação. Gerada pelo notebook 05. Três invariantes esperam zero; a da ponte N:N espera valor maior que zero, por construção.';

ALTER TABLE gold.relatorio_invariantes ALTER COLUMN invariante COMMENT
'Regra verificada: reconciliação, total estadual, integridade referencial ou ponte N:N.';
ALTER TABLE gold.relatorio_invariantes ALTER COLUMN objeto COMMENT 'Tabela ou par de tabelas avaliado.';
ALTER TABLE gold.relatorio_invariantes ALTER COLUMN valor COMMENT 'Resultado numérico da verificação.';
ALTER TABLE gold.relatorio_invariantes ALTER COLUMN esperado COMMENT 'Critério de aceitação.';
ALTER TABLE gold.relatorio_invariantes ALTER COLUMN _execucao_ts COMMENT 'Momento da verificação.';

##15. Extração do catálogo

In [0]:
%sql

-- catálogo de tabelas

SELECT table_name AS tabela, comment AS descricao
FROM mvp_pipeline.information_schema.tables
WHERE table_schema = 'gold'
ORDER BY
  CASE WHEN table_name LIKE 'dim_%' THEN 1
       WHEN table_name LIKE 'ponte_%' THEN 2
       ELSE 3 END,
  table_name;

In [0]:
%sql

-- dicionário de dados: toda coluna de toda tabela da Gold

SELECT table_name AS tabela,
       ordinal_position AS posicao,
       column_name AS coluna,
       full_data_type AS tipo,
       comment AS descricao
FROM mvp_pipeline.information_schema.columns
WHERE table_schema = 'gold'
ORDER BY table_name, ordinal_position;

In [0]:
%sql

-- controle: colunas ainda sem comentário

SELECT table_name, column_name
FROM mvp_pipeline.information_schema.columns
WHERE table_schema = 'gold' AND (comment IS NULL OR comment = '')
ORDER BY table_name, ordinal_position;